####Text To Image Generation
####Google Gemini Integration using Langchain API
<!-- https://reference.langchain.com/python/langchain-google-genai/chat_models/ChatGoogleGenerativeAI?_gl=1*b2b0al*_gcl_au*MTQyODM2MzA3Ni4xNzc4NDMzODM2*_ga*MTc2NjU1ODg4Ni4xNzc4NDMzODM4*_ga_47WX3HKKY2*czE3Nzk1Mzg3NTMkbzE0JGcxJHQxNzc5NTM4ODkxJGo2MCRsMCRoMA..

https://docs.langchain.com/oss/python/integrations/chat/google_generative_ai?_gl=1*tzk6mc*_gcl_au*MTQyODM2MzA3Ni4xNzc4NDMzODM2*_ga*MTc2NjU1ODg4Ni4xNzc4NDMzODM4*_ga_47WX3HKKY2*czE3Nzk1Mzg3NTMkbzE0JGcxJHQxNzc5NTM4OTAyJGo0OSRsMCRoMA..#image-generation -->

In [ ]:
import os
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import AIMessage
import base64
from IPython.display import Image, display
import requests  # ← add this line


Google_API_Token_Key=os.getenv("GOOGLE_API_KEY")
if(Google_API_Token_Key is None):
    raise ValueError("GOOGLE_API_KEY environment variable not set.")
else:
 key=   os.environ["GOOGLE_API_KEY"] = Google_API_Token_Key;

# # Direct API test
# url = f"https://generativelanguage.googleapis.com/v1beta/models?key={key}"
# response = requests.get(url)

# print(f"\nAPI Test Status: {response.status_code}")
# if response.status_code == 200:
#     print("✅ Key is VALID")
# else:
#     print(f"❌ Key INVALID: {response.json()}")

model=ChatGoogleGenerativeAI(model="gemini-2.5-flash-image",google_api_key=key)

response=model.invoke("Generative a dog Image",generation_config={
        "response_modalities": ["TEXT", "IMAGE"] 
    })

def get_generate_image(response, AIMessage)->None:
    image_block = next(
        block
        for block in response.content
        if isinstance(block, dict) and block.get("image_url")
    )
    return image_block["image_url"].get("url").split(",")[-1]

image_base64 = get_generate_image(response)
display(Image(data=base64.b64decode(image_base64), width=300))

model = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash-image",
    image_config={"aspect_ratio": "16:9"},
)

   


ChatGoogleGenerativeAIError: Error calling model 'gemini-2.5-flash-image-preview' (NOT_FOUND): 404 NOT_FOUND. {'error': {'code': 404, 'message': 'models/gemini-2.5-flash-image-preview is not found for API version v1beta, or is not supported for generateContent. Call ModelService.ListModels to see the list of available models and their supported methods.', 'status': 'NOT_FOUND'}}

#Text To Image With hugging Model provider Using Lanchain Framework

In [15]:
import os
from dotenv import load_dotenv
from huggingface_hub import InferenceClient
from langchain.tools import tool
from IPython.display import Image as IPImage, display

# ============================================
# Step 1: Load environment variables
# ============================================
load_dotenv(override=True)

HF_TOKEN = os.getenv("HUGGINGFACEHUB_API_TOKEN")
if not HF_TOKEN:
    raise ValueError("❌ HUGGINGFACEHUB_API_TOKEN not found in .env")


# ============================================
# Step 2: Create the LangChain Tool
# ============================================
@tool
def generate_image(prompt: str) -> str:
    """
    Generates an image from a text prompt using HuggingFace Stable Diffusion.    
    Args:
        prompt: Text description of the image to generate        
    Returns:
        Path to the saved image file
    """
    try:
        # Initialize HuggingFace client
        client = InferenceClient(token=HF_TOKEN)        
        print(f"Generating image for: '{prompt}'...")        
        # Generate image
        image = client.text_to_image(
            prompt=prompt,
            model="stabilityai/stable-diffusion-3-medium"
        )        
        # Save image
        output_path = "generated_image.png"
        image.save(output_path)        
        print(f"✅ Image saved to: {output_path}")
        return output_path        
    except Exception as e:
        error_msg = f"Image generation failed: {str(e)}"
        print(error_msg)
        return error_msg


# ============================================
# Step 3: Use the Tool (Direct Invocation)
# ============================================
if __name__ == "__main__":
    
    # Method 1: Direct invocation using .invoke()
    print("\n" + "="*50)
    print("Method 1: Direct Tool Invocation")
    print("="*50)
    
    result = generate_image.invoke("Forest Image with water stream")
    print(f"Result: {result}")
    
    # Display image in Jupyter Notebook
    if os.path.exists(result):
        display(IPImage(filename=result, width=400))


Method 1: Direct Tool Invocation
Generating image for: 'Forest Image with water stream'...
Image generation failed: Client error '402 Payment Required' for url 'https://router.huggingface.co/fal-ai/fal-ai/stable-diffusion-v3-medium' (Request ID: Root=1-6a12b2d5-562649bd5071c23012e24a67;117417b3-ca51-4225-bf16-bd4f7d0a23df)
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/402

You have depleted your monthly included credits. Purchase pre-paid credits to continue using Inference Providers. Alternatively, subscribe to PRO to get 20x more included usage.
Result: Image generation failed: Client error '402 Payment Required' for url 'https://router.huggingface.co/fal-ai/fal-ai/stable-diffusion-v3-medium' (Request ID: Root=1-6a12b2d5-562649bd5071c23012e24a67;117417b3-ca51-4225-bf16-bd4f7d0a23df)
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/402

You have depleted your monthly included credits. Purchase pre-paid credits